In [1]:
# ==========================================================
# Notebook 02: ACL and Metadata Indexing
# ==========================================================

import random
import ast

import pandas as pd
import numpy as np

In [2]:
DATA_PATH = "data/processed/enterprise_corpus.csv"

enterprise_df = pd.read_csv(DATA_PATH)

enterprise_df.head()

,source,title,content,author,doc_id,department,created_date,tags
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']"
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']"
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']"
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']"
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']"


In [3]:
print("Columns:")

print(enterprise_df.columns.tolist())

Columns:
['source', 'title', 'content', 'author', 'doc_id', 'department', 'created_date', 'tags']


In [4]:
roles = ["Engineering", "HR", "Finance", "Legal", "Public"]

In [5]:
def generate_acl(department):

    if department == "Engineering":

        return ["Engineering", "Public"]

    elif department == "HR":

        return ["HR", "Public"]

    elif department == "Finance":

        return ["Finance", "Public"]

    else:

        return ["Legal", "Public"]

In [6]:
enterprise_df["acl_roles"] = enterprise_df["department"].apply(generate_acl)

enterprise_df[["doc_id", "department", "acl_roles"]]

,doc_id,department,acl_roles
0,DOC0001,Engineering,"[Engineering, Public]"
1,DOC0002,Finance,"[Finance, Public]"
2,DOC0003,HR,"[HR, Public]"
3,DOC0004,Engineering,"[Engineering, Public]"
4,DOC0005,Engineering,"[Engineering, Public]"
5,DOC0006,HR,"[HR, Public]"


In [7]:
security_mapping = {
    "Engineering": "Internal",
    "HR": "Restricted",
    "Finance": "Restricted",
    "Legal": "Internal",
}

enterprise_df["security_level"] = enterprise_df["department"].map(security_mapping)

In [8]:
enterprise_df["owner_team"] = enterprise_df["department"]

In [9]:
enterprise_df.head()

,source,title,content,author,doc_id,department,created_date,tags,acl_roles,security_level,owner_team
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']","[Engineering, Public]",Internal,Engineering
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']","[Finance, Public]",Restricted,Finance
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']","[HR, Public]",Restricted,HR
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']","[Engineering, Public]",Internal,Engineering
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']","[Engineering, Public]",Internal,Engineering


In [10]:
enterprise_df.head()

,source,title,content,author,doc_id,department,created_date,tags,acl_roles,security_level,owner_team
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']","[Engineering, Public]",Internal,Engineering
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']","[Finance, Public]",Restricted,Finance
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']","[HR, Public]",Restricted,HR
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']","[Engineering, Public]",Internal,Engineering
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']","[Engineering, Public]",Internal,Engineering


In [11]:
users = {"alice": "Engineering", "bob": "HR", "charlie": "Finance", "diana": "Legal"}

users

{'alice': 'Engineering', 'bob': 'HR', 'charlie': 'Finance', 'diana': 'Legal'}

In [12]:
def acl_filter(dataframe, user_department):

    mask = dataframe["acl_roles"].apply(lambda roles: user_department in roles)

    return dataframe[mask].reset_index(drop=True)

In [13]:
engineering_docs = acl_filter(enterprise_df, "Engineering")

engineering_docs[["doc_id", "title", "acl_roles"]]

,doc_id,title,acl_roles
0,DOC0001,Phoenix Deployment Discussion,"[Engineering, Public]"
1,DOC0004,Enterprise Search Design,"[Engineering, Public]"
2,DOC0005,PHX-245 Database Migration,"[Engineering, Public]"


In [14]:
hr_docs = acl_filter(enterprise_df, "HR")

hr_docs[["doc_id", "title", "acl_roles"]]

,doc_id,title,acl_roles
0,DOC0003,Phoenix Architecture Wiki,"[HR, Public]"
1,DOC0006,SEC-104 ACL Enhancement,"[HR, Public]"


In [15]:
acl_summary = (
    enterprise_df.explode("acl_roles")
    .groupby("acl_roles")
    .size()
    .reset_index(name="document_count")
)

acl_summary

,acl_roles,document_count
0,Engineering,3
1,Finance,1
2,HR,2
3,Public,6


In [16]:
filtered_docs = enterprise_df[
    (enterprise_df["department"] == "Engineering")
    & (enterprise_df["created_date"] >= "2026-01-12")
]

filtered_docs[["doc_id", "title", "created_date"]]

,doc_id,title,created_date
3,DOC0004,Enterprise Search Design,2026-01-14
4,DOC0005,PHX-245 Database Migration,2026-01-15


In [17]:
OUTPUT_PATH = "data/processed/"

enterprise_df.to_csv(OUTPUT_PATH + "enterprise_corpus_acl.csv", index=False)

print("ACL-enabled corpus saved!")

ACL-enabled corpus saved!


In [18]:
loaded_df = pd.read_csv("data/processed/enterprise_corpus_acl.csv")

loaded_df.head()

,source,title,content,author,doc_id,department,created_date,tags,acl_roles,security_level,owner_team
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']","['Engineering', 'Public']",Internal,Engineering
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']","['Finance', 'Public']",Restricted,Finance
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']","['HR', 'Public']",Restricted,HR
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']","['Engineering', 'Public']",Internal,Engineering
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']","['Engineering', 'Public']",Internal,Engineering
